# From a Data Explorer cohort to a notebook

Data Explorer builds a cohort with no code. When you **Export** a data snapshot, Data Explorer writes
the selected tables as CSV files (zipped or unzipped) to a workspace bucket you choose. This notebook
picks those files up so an analyst can continue in Python.

**Before running:** in Data Explorer, build a cohort on Project Baseline, select the tables to export,
choose the `shared-work` bucket, and note the folder it created. Set `EXPORT_PREFIX` below to that folder.

**Time:** about 2 minutes.

In [ ]:
import io
import subprocess
import zipfile

import pandas as pd
from google.cloud import storage


def resolve(resource_name):
    return subprocess.run(["wb", "resolve", "--name", resource_name], capture_output=True, text=True, check=True).stdout.strip()


BUCKET_RESOURCE = "shared-work"
EXPORT_PREFIX = ""          # e.g. "data-explorer-exports/my-asthma-cohort"; leave empty to list what's in the bucket

bucket_name, _, base_prefix = resolve(BUCKET_RESOURCE).removeprefix("gs://").partition("/")
bucket = storage.Client().bucket(bucket_name)
search_prefix = "/".join(p for p in (base_prefix, EXPORT_PREFIX) if p)

## 1. Find the export

In [ ]:
blobs = [b for b in bucket.list_blobs(prefix=search_prefix) if b.name.lower().endswith((".csv", ".zip"))]
listing = pd.DataFrame({"file": [b.name for b in blobs], "size_mb": [round(b.size / 1e6, 2) for b in blobs],
                        "updated": [b.updated.strftime("%Y-%m-%d %H:%M") for b in blobs]})
listing

## 2. Load every table in the export
Zipped exports are unpacked in memory; nothing is written to the app's disk.

In [ ]:
tables = {}
for b in blobs:
    data = b.download_as_bytes()
    if b.name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for member in z.namelist():
                if member.lower().endswith(".csv"):
                    tables[member.rsplit("/", 1)[-1].removesuffix(".csv")] = pd.read_csv(z.open(member), low_memory=False)
    else:
        tables[b.name.rsplit("/", 1)[-1].removesuffix(".csv")] = pd.read_csv(io.BytesIO(data), low_memory=False)

pd.DataFrame({"table": list(tables), "rows": [len(t) for t in tables.values()], "columns": [t.shape[1] for t in tables.values()]})

## 3. Pick up where Data Explorer left off
The participant key is the first column that looks like an ID (`USUBJID` for Project Baseline).

In [ ]:
if tables:
    first = next(iter(tables.values()))
    id_col = next((c for c in first.columns if c.upper() in ("USUBJID", "PERSON_ID", "SUBJID")), first.columns[0])
    print(f"{first[id_col].nunique():,} participants in the cohort (key: {id_col})")
    display(first.head())
else:
    print("No CSV files found. Check EXPORT_PREFIX against the folder Data Explorer created in the bucket.")

From here the cohort can be joined to any other Baseline table in BigQuery on the same key, saved as
Parquet to the bucket for the R users, or handed to the Workbench AI assistant ("characterize the cohort in
`examples/…`").

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd